# Selecting OTel GenAI Semantic Convention Attributes in TruLens

Short answer: **yes, for `gen_ai.*` span attributes.**

TruLens emits [OpenTelemetry GenAI semantic convention](https://opentelemetry.io/docs/specs/semconv/gen-ai/)
attributes alongside its own `ai.observability.*` attributes, and
`Selector.span_attribute` is an unvalidated raw attribute-name lookup. So you
can point any metric at a `gen_ai.*` key directly, without TruLens-specific
attribute knowledge.

Every metric below is a built-in TruLens evaluator — the only thing that changes
is the selector pointing it at a `gen_ai.*` attribute:

| Selector mode | Metric here |
| --- | --- |
| Per-item over a list attribute (`collect_list=False`) | Context Relevance |
| Whole list at once (`collect_list=True`) | Groundedness |
| Whole trace (`trace_level=True`) | Logical Consistency |
| Whole conversation (`.on_conversation()`) | Conversation Helpfulness |

For a worked coding-agent example, evaluating a Claude Code session assembled
from client hooks, see [`coding_agent_trace_evaluation.ipynb`](./coding_agent_trace_evaluation.ipynb).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/truera/trulens/blob/main/examples/expositional/otel_genai/genai_semconv_attribute_selection.ipynb)

## What TruLens emits, and what it does not

`@instrument()` sets `gen_ai.*` attributes automatically based on `span_type`.
You do not call any GenAI-specific API — you set the TruLens attributes as
usual and the `gen_ai.*` mirror is written for you.

| `span_type` | `gen_ai.*` span attributes emitted |
| --- | --- |
| `GENERATION` | `gen_ai.operation.name`, `gen_ai.request.model`, `gen_ai.request.temperature`, `gen_ai.system`, `gen_ai.usage.input_tokens`, `gen_ai.usage.output_tokens` |
| `RETRIEVAL` | `gen_ai.retrieval.query.text`, `gen_ai.retrieval.documents` |
| `TOOL`, `MCP` | `gen_ai.tool.name`, `gen_ai.tool.call.arguments`, `gen_ai.tool.call.result` |

Message content is handled separately from metadata. Prompts and completions are
emitted as a `gen_ai.client.inference.operation.details` span *event* rather than
as span attributes, gated behind `TRULENS_OTEL_CAPTURE_CONTENT` for PII safety.
Every selector in this notebook reads span attributes, so it uses
`ai.observability.record_root.input` / `.output` — or the function-call
attributes — when it needs the prompt or the response.

One limit is worth knowing before you design around this: **spans from
third-party instrumentation are dropped.** TruLens' exporter keeps only spans
carrying `ai.observability.app_name`, so spans produced by e.g. `openinference`
or `opentelemetry-instrumentation-openai` are filtered out at export and never
become selectable. `gen_ai.*` selection works on spans TruLens itself produced.

`gen_ai.retrieval.query.text` and `gen_ai.retrieval.documents` are TruLens
aliases under the `gen_ai` namespace, not official OTel GenAI attributes — the
spec has no retrieval attributes yet.

## Setup

In [1]:
# !pip install trulens trulens-providers-cortex snowflake-snowpark-python

In [2]:
import os

from snowflake.cortex import complete
from snowflake.snowpark import Session
from trulens.apps.app import TruApp
from trulens.core import Metric
from trulens.core import TruSession
from trulens.core.database.connector.default import DefaultDBConnector
from trulens.core.feedback.selector import Selector
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import GenAIAttributes
from trulens.otel.semconv.trace import SpanAttributes
from trulens.providers.cortex import Cortex

Package jsonschema not present in requirements.


Connect to Snowflake. `APP_MODEL` generates answers; `JUDGE_MODEL` runs the
built-in evaluators.

In [3]:
snowpark_session = Session.builder.config(
    "connection_name", os.environ.get("SNOWFLAKE_CONNECTION_NAME", "default")
).create()

APP_MODEL = "llama3.3-70b"
JUDGE_MODEL = "claude-sonnet-4-5"

provider = Cortex(snowpark_session=snowpark_session, model_engine=JUDGE_MODEL)

session = TruSession(
    connector=DefaultDBConnector(database_url="sqlite:///genai_semconv_demo.sqlite")
)
session.reset_database()

## An instrumented support agent

Three instrumented methods, one span type each. Note that nothing here mentions
`gen_ai` — these are the ordinary TruLens attributes, and the `gen_ai.*` mirror
is derived from them. The `GENERATION` span picks up `gen_ai.request.model` from
`COST.MODEL`, plus `temperature`, `provider_name`, and `operation_name`, which
the instrumentor looks for by name.

In [4]:
KB = [
    "Password resets are self-serve from Account -> Security -> Reset password.",
    "A reset link stays valid for 30 minutes, then must be requested again.",
    "Resetting a password signs the account out of every active session.",
    "Refunds for annual plans are prorated from the cancellation date.",
    "Our headquarters is in Bellevue, Washington.",
]


class SupportAgent:
    @instrument(
        span_type=SpanAttributes.SpanType.RETRIEVAL,
        attributes={
            SpanAttributes.RETRIEVAL.QUERY_TEXT: "query",
            SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
        },
    )
    def retrieve(self, query: str) -> list:
        """Keyword overlap stand-in for a vector search."""
        terms = {t for t in query.lower().split() if len(t) > 3}
        scored = [(len(terms & set(doc.lower().split())), doc) for doc in KB]
        scored.sort(key=lambda pair: pair[0], reverse=True)
        return [doc for _, doc in scored[:3]]

    @instrument(
        span_type=SpanAttributes.SpanType.GENERATION,
        attributes=lambda ret, exception, *args, **kwargs: {
            SpanAttributes.COST.MODEL: APP_MODEL,
            # Plain keys the instrumentor maps into gen_ai.* by name.
            "temperature": 0.0,
            "provider_name": "snowflake.cortex",
            "operation_name": "chat",
        },
    )
    def generate(self, query: str, contexts: list) -> str:
        prompt = (
            "Answer the customer's question using only the context below. "
            "Be concise.\n\nContext:\n"
            + "\n".join(f"- {c}" for c in contexts)
            + f"\n\nQuestion: {query}"
        )
        return complete(
            APP_MODEL, [{"role": "user", "content": prompt}], session=snowpark_session
        ).strip()

    @instrument(
        span_type=SpanAttributes.SpanType.RECORD_ROOT,
        attributes={
            SpanAttributes.RECORD_ROOT.INPUT: "query",
            SpanAttributes.RECORD_ROOT.OUTPUT: "return",
        },
    )
    def answer(self, query: str) -> str:
        return self.generate(query=query, contexts=self.retrieve(query))

## Metrics

Four built-in evaluators, each pointed at a `gen_ai.*` attribute by its selector.

`Selector(span_attribute=...)` is a raw attribute lookup with no whitelist, which
is why arbitrary OTel keys work. The same applies to a scalar attribute or a
derived value, neither of which needs a custom metric to be useful:

```python
# Any single gen_ai.* key.
Selector(
    span_type=SpanAttributes.SpanType.GENERATION,
    span_attribute=GenAIAttributes.REQUEST.MODEL,
)

# Or reshape several of them before the metric sees them.
Selector(
    span_type=SpanAttributes.SpanType.GENERATION,
    span_attributes_processor=lambda attrs: attrs.get(
        GenAIAttributes.USAGE.INPUT_TOKENS, 0
    ),
)
```

### Per-item selection — Context Relevance

`collect_list=False` fans the metric out over each element of
`gen_ai.retrieval.documents`, scoring one document per call, then averages. The
query comes from one `gen_ai.*` attribute and each document from another, both on
the same `RETRIEVAL` span.

### Whole-list selection — Groundedness

Same attribute, `collect_list=True`: all documents arrive as one list so the
judge can check the answer against the full evidence set at once.

### Trace and conversation scope

`trace_level=True` hands the metric a `Trace` covering every span in the record
instead of one span's attribute, and must be the only selector on that metric.
`.on_conversation()` groups every `RECORD_ROOT` sharing a `conversation_id`,
ordered by start time, and attaches its score to the conversation's last record.

In [5]:
m_context_relevance = Metric(
    implementation=provider.context_relevance_with_cot_reasons,
    name="Context Relevance",
).on({
    "question": Selector(
        span_type=SpanAttributes.SpanType.RETRIEVAL,
        span_attribute=GenAIAttributes.RETRIEVAL.QUERY_TEXT,
    ),
    "context": Selector(
        span_type=SpanAttributes.SpanType.RETRIEVAL,
        span_attribute=GenAIAttributes.RETRIEVAL.DOCUMENTS,
        collect_list=False,  # one LLM call per retrieved document
    ),
})

m_groundedness = Metric(
    implementation=provider.groundedness_measure_with_cot_reasons,
    name="Groundedness",
).on({
    "source": Selector(
        span_type=SpanAttributes.SpanType.RETRIEVAL,
        span_attribute=GenAIAttributes.RETRIEVAL.DOCUMENTS,
        collect_list=True,  # one LLM call against all documents
    ),
    "statement": Selector.select_record_output(),
})

m_answer_relevance = Metric(
    implementation=provider.relevance_with_cot_reasons,
    name="Answer Relevance",
).on({
    "prompt": Selector.select_record_input(),
    "response": Selector.select_record_output(),
})

m_logical_consistency = Metric(
    implementation=provider.logical_consistency_with_cot_reasons,
    name="Logical Consistency",
).on({"trace": Selector(trace_level=True)})

m_conversation_helpfulness = Metric(
    implementation=provider.conversation_helpfulness_with_cot_reasons,
    name="Conversation Helpfulness",
).on_conversation()

## Record a conversation

Passing `conversation_id` to the recording context is what makes the
conversation-level metric possible.

In [6]:
agent = SupportAgent()
recorder = TruApp(
    agent,
    app_name="GenAI Semconv Selection",
    app_version=APP_MODEL,
    main_method=agent.answer,
    feedbacks=[
        m_context_relevance,
        m_groundedness,
        m_answer_relevance,
        m_logical_consistency,
        m_conversation_helpfulness,
    ],
)

turns = [
    "How do I reset my password?",
    "How long does that reset link stay valid?",
    "Will resetting sign me out everywhere?",
]

with recorder(conversation_id="support-thread-1") as recording:
    for turn in turns:
        print(f"Q: {turn}")
        print(f"A: {agent.answer(turn)}\n")

session.force_flush()

Q: How do I reset my password?


A: To reset your password, go to Account -> Security -> Reset password. This is a self-serve process.

Q: How long does that reset link stay valid?


A: 30 minutes.

Q: Will resetting sign me out everywhere?


A: Yes, resetting your password will sign your account out of every active session.



True

## Compute and inspect the metrics

In [7]:
recorder.compute_feedbacks(raise_error_on_no_feedbacks_computed=False)
session.force_flush()

/Users/jreini/Desktop/development/git-sfc/trulens/src/feedback/trulens/feedback/llm_provider.py:3407: UserWarning: Failed to process and remove trivial statements. Proceeding with all statements.
  hypotheses = self._remove_trivial_statements(hypotheses)


True

In [8]:
records, metric_names = session.get_records_and_feedback(
    app_name="GenAI Semconv Selection"
)

present = [name for name in metric_names if name in records.columns]
records[["input"] + present].round(3)

,input,Context Relevance,Groundedness,Answer Relevance,Logical Consistency,Conversation Helpfulness
0,How do I reset my password?,0.556,1.000,1.0,0.333,NaN
1,How long does that reset link stay valid?,0.556,0.667,1.0,0.333,NaN
2,Will resetting sign me out everywhere?,0.556,1.000,1.0,0.333,0.667


## Dashboard

In [ ]:
from trulens.dashboard import run_dashboard

run_dashboard(session)